# E-commerce Sales - Data Cleaning

**Objective:** Clean and prepare the raw data for analysis.

**Input:** Raw transaction data  
**Output:** Cleaned dataset ready for analysis

**Author:** Hamdi Ahmed 
**Date:** 20/06/2026

# 1. Setup

In [68]:
import duckdb
import pandas as pd

df1 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df2 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')
raw = pd.concat([df1, df2], ignore_index=True)


print(f"Loaded {len(raw):,} rows")

Loaded 1,067,371 rows


# 2. Standardise Column Names

In [69]:
raw = raw.rename(columns={'Customer ID': 'CustomerID'})

con = duckdb.connect()
con.register('raw_retail', raw)

print("Columns:", list(raw.columns))
print(f"Rows: {len(raw):,}")

Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country']
Rows: 1,067,371


# 3. Clean Up Stock Code

In [70]:
con.execute("""
   SELECT
        LENGTH(CAST(StockCode AS VARCHAR))  AS code_length,
        COUNT(*)                            AS rows,
        COUNT(DISTINCT StockCode)           AS distinct_codes,
        ANY_VALUE(StockCode)                AS example
    FROM raw_retail
    GROUP BY code_length
    ORDER BY rows DESC
""").fetchdf()

,code_length,rows,distinct_codes,example
0,5,932385,3590,22581
1,6,127590,1649,16161M
2,4,2158,4,POST
3,1,1713,5,M
4,3,1446,1,DOT
5,7,1393,9,15056BL
6,2,283,2,C2
7,12,202,10,gift_0001_20
8,8,127,30,DCGS0076
9,9,74,5,DCGSSGIRL


In [72]:
# Remove StockCodes with length less than 6 or bank charges

stock_code_clean = con.execute("""
   SELECT *
   FROM 
       (SELECT *
           FROM raw_retail
           WHERE
               LENGTH(StockCode) >=6
        )
    WHERE StockCode <> 'BANK CHARGES'
""").fetchdf()

In [73]:
con.execute("""
   SELECT
        LENGTH(CAST(StockCode AS VARCHAR))  AS code_length,
        COUNT(*)                            AS rows,
        COUNT(DISTINCT StockCode)           AS distinct_codes,
        ANY_VALUE(StockCode)                AS example
    FROM stock_code_clean
    GROUP BY code_length
    ORDER BY rows DESC
""").fetchdf()

,code_length,rows,distinct_codes,example
0,6,127590,1649,51014C
1,7,1393,9,15056bl
2,8,127,30,DCGSSBOY
3,12,100,9,gift_0001_20
4,9,74,5,AMAZONFEE


In [87]:
# Remove Amazon Fee

stock_code_cleaned = con.execute("""
   SELECT *
   FROM 
       stock_code_clean
    WHERE
        StockCode <> 'AMAZONFEE'
""").fetchdf()

In [98]:
con.execute("""
   SELECT
        LENGTH(CAST(StockCode AS VARCHAR))  AS code_length,
        COUNT(*)                            AS rows,
        COUNT(DISTINCT StockCode)           AS distinct_codes,
        ANY_VALUE(StockCode)                AS example
    FROM stock_code_cleaned
    GROUP BY code_length
    ORDER BY rows DESC
""").fetchdf()

,code_length,rows,distinct_codes,example
0,6,127590,1649,85123A
1,7,1393,9,15056bl
2,8,127,30,DCGSSBOY
3,12,100,9,gift_0001_20
4,9,31,4,DCGSSGIRL


In [107]:
con.execute("""
SELECT
    CAST(StockCode AS VARCHAR)          AS code,
    ANY_VALUE(Description)              AS description,
    COUNT(*)                            AS rows,
    ROUND(MIN(Price), 2)                AS min_price,
    ROUND(MAX(Price), 2)                AS max_price,
    ROUND(SUM(Quantity * Price), 2)     AS revenue
FROM raw_retail
WHERE NOT REGEXP_MATCHES(CAST(StockCode AS VARCHAR), '[0-9]')
GROUP BY code
ORDER BY revenue DESC
""").fetchdf()

,code,description,rows,min_price,max_price,revenue
0,DOT,DOTCOM POSTAGE,1446,0.00,4505.17,322647.47
1,POST,POSTAGE,2122,0.00,8142.75,112341.00
2,ADJUST,Adjustment by Peter on 24/05/2010 1,67,4.57,5117.03,6835.24
3,DCGSSGIRL,GIRLS PARTY BAG,25,0.00,3.36,295.63
4,DCGSSBOY,BOYS PARTY BAG,23,0.00,3.36,251.35
5,m,Manual,5,2.55,4.00,15.05
6,GIFT,NaN,1,0.00,0.00,0.00
7,DCGSLGIRL,NaN,1,0.00,0.00,0.00
8,DCGSLBOY,NaN,1,0.00,0.00,0.00
9,PADS,PADS TO MATCH ALL CUSHIONS,19,0.00,36.60,-36.58


## • Building CTE chain to clean raw data in stages, with each stage fixing one issue found in audit. 
## • Some observations will have more than 1 of the data quality issues identified. 
## • Before each stage, the number of data quality issues remaining will be counted to check if they have been removed in the previous stage

# 3. Remove Exact Duplicates

In [88]:
# Audit found 34,335 exact duplicate rows where all columns including the InvoiceDate timestamp are identical
# Likely data extraction issue as same transaction can't have happened twice.

con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM stock_code_cleaned
    )
    SELECT COUNT(*) AS no_duplication_total FROM deduped;
""").fetchdf()


,no_duplication_total
0,125462


# 4. Remove Cancellations

#### Audit found 19,494 cancellations with C-prefix in invoice.

In [89]:
# Remaining data quality issues
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * FROM stock_code_cleaned
    )
    SELECT
        COUNT(*) total_rows,
        SUM(CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' THEN 1 ELSE 0 END) AS cancellations,
        SUM(CASE WHEN Quantity <= 0 THEN 1 ELSE 0 END) AS invalid_quantity,
        SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS invalid_price,
        SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description
    FROM deduped;
""").fetchdf()

,total_rows,cancellations,invalid_quantity,invalid_price,null_customer_id,null_description
0,125462,2230.0,3407.0,1811.0,37242.0,1480.0


In [90]:
# Remove Cancellations
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM stock_code_cleaned
    ),
    no_cancellations AS (
        SELECT * FROM deduped
        WHERE CAST(Invoice AS VARCHAR) NOT LIKE 'C%'
    )
    SELECT COUNT(*) AS rows_after_filter FROM no_cancellations;
""").fetchdf()

,rows_after_filter
0,123232


In [91]:
# Remaining data quality issues
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM stock_code_cleaned
    ),
    no_cancellations AS (
        SELECT * FROM deduped
        WHERE CAST(Invoice AS VARCHAR) NOT LIKE 'C%'
    )
    SELECT
        COUNT(*) total_rows,
        SUM(CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' THEN 1 ELSE 0 END) AS cancellations,
        SUM(CASE WHEN Quantity <= 0 THEN 1 ELSE 0 END) AS invalid_quantity,
        SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS invalid_price,
        SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description
    FROM no_cancellations;
""").fetchdf()

,total_rows,cancellations,invalid_quantity,invalid_price,null_customer_id,null_description
0,123232,0.0,1177.0,1811.0,37192.0,1480.0


# 5. Remove negative quantities and zero/negative prices

#### Audit found 22,950 with negative quantities and 6,207 with zero/negative prices

In [92]:
# remove zero/negative quantities and prices

con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM stock_code_cleaned
    ),
    no_cancellations AS (
        SELECT * FROM deduped
        WHERE CAST(Invoice AS VARCHAR) NOT LIKE 'C%'
    ),
    base_transactions AS (
        SELECT * 
        FROM no_cancellations
        WHERE Quantity > 0 AND Price > 0
    )
    SELECT COUNT(*) AS rows_after_filter FROM base_transactions;
""").fetchdf()

,rows_after_filter
0,121421


In [93]:
# Remaining data quality issues
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM stock_code_cleaned
    ),
    no_cancellations AS (
        SELECT * FROM deduped
        WHERE CAST(Invoice AS VARCHAR) NOT LIKE 'C%'
    ),
    base_transactions AS (
        SELECT * 
        FROM no_cancellations
        WHERE Quantity > 0 AND Price > 0
    )
    SELECT
        COUNT(*) total_rows,
        SUM(CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' THEN 1 ELSE 0 END) AS cancellations,
        SUM(CASE WHEN Quantity <= 0 THEN 1 ELSE 0 END) AS invalid_quantity,
        SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS invalid_price,
        SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description
    FROM base_transactions;
""").fetchdf()

,total_rows,cancellations,invalid_quantity,invalid_price,null_customer_id,null_description
0,121421,0.0,0.0,0.0,35385.0,0.0


# 6. Tag Guests

#### • 228,488 observations (22.7% of total) remaining have no CustomerID, their revenue is valid but will be filtered out for the customer analysis
#### • Null values in Description were found in audit but have been removed in stage 5

In [94]:
# Tag rows with no CustomerID as guest transactions
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM stock_code_cleaned
    ),
    no_cancellations AS (
        SELECT * FROM deduped
        WHERE CAST(Invoice AS VARCHAR) NOT LIKE 'C%'
    ),
    base_transactions AS (
        SELECT * 
        FROM no_cancellations
        WHERE Quantity > 0 AND Price > 0
    ),
    guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM base_transactions
    )
    SELECT
        SUM(is_guest) AS guest_rows,
        SUM(1 - is_guest) AS registered_rows,
        COUNT(*) AS total_rows
    FROM guest_tag;
    """).fetchdf()

,guest_rows,registered_rows,total_rows
0,35385.0,86036.0,121421


In [95]:
# Remaining data quality issues

con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM stock_code_cleaned
    ),
    no_cancellations AS (
        SELECT * FROM deduped
        WHERE CAST(Invoice AS VARCHAR) NOT LIKE 'C%'
    ),
    base_transactions AS (
        SELECT * 
        FROM no_cancellations
        WHERE Quantity > 0 AND Price > 0
    ),
    guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM base_transactions
    )
    SELECT
        COUNT(*) total_rows,
        SUM(CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' THEN 1 ELSE 0 END) AS cancellations,
        SUM(CASE WHEN Quantity <= 0 THEN 1 ELSE 0 END) AS invalid_quantity,
        SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS invalid_price,
        SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
        SUM(is_guest) AS guest_total,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description
    FROM guest_tag;
""").fetchdf()

,total_rows,cancellations,invalid_quantity,invalid_price,null_customer_id,guest_total,null_description
0,121421,0.0,0.0,0.0,35385.0,35385.0,0.0


# 6. Derive Revenue

In [96]:
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM stock_code_cleaned
    ),
    no_cancellations AS (
        SELECT * FROM deduped
        WHERE CAST(Invoice AS VARCHAR) NOT LIKE 'C%'
    ),
    base_transactions AS (
        SELECT * 
        FROM no_cancellations
        WHERE Quantity > 0 AND Price > 0
    ),
    guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM base_transactions
    ),
    revenue_derive AS (
        SELECT 
            *,
            Quantity * Price AS Revenue
        FROM guest_tag
    )
    SELECT
        MIN(Revenue) AS min_revenue,
        MAX(Revenue) AS max_revenue,
        ROUND(AVG(Revenue), 2) AS avg_revenue,
        ROUND(SUM(Revenue), 2) AS total_revenue,
        MIN(Quantity) AS min_quantity,
        MAX(Quantity) AS max_quantity,
        ROUND(AVG(Quantity), 2) AS avg_quantity,
        ROUND(SUM(Quantity), 2) AS total_quantity,
        MIN(Price) AS min_price,
        MAX(Price) AS max_price,
        ROUND(AVG(Price), 2) AS avg_price,
        ROUND(SUM(Price), 2) AS total_price,
        COUNT(*) total_rows
    FROM revenue_derive;
    """).fetchdf()

,min_revenue,max_revenue,avg_revenue,total_revenue,min_quantity,max_quantity,avg_quantity,total_quantity,min_price,max_price,avg_price,total_price,total_rows
0,0.12,8925.0,19.6,2379343.47,1,6696,9.83,1193279.0,0.05,5117.03,3.52,427940.24,121421


In [101]:
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM stock_code_cleaned
    ),
    no_cancellations AS (
        SELECT * FROM deduped
        WHERE CAST(Invoice AS VARCHAR) NOT LIKE 'C%'
    ),
    base_transactions AS (
        SELECT * 
        FROM no_cancellations
        WHERE Quantity > 0 AND Price > 0
    ),
    guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM base_transactions
    ),
    revenue_derive AS (
        SELECT 
            *,
            Quantity * Price AS Revenue
        FROM guest_tag
    )
    SELECT
        *
    FROM revenue_derive
    WHERE Price = 5117.03	
    """).fetchdf()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,is_guest,Revenue
0,495798,ADJUST,Adjustment by john on 26/01/2010 17,1,2010-01-26 17:25:00,5117.03,NaN,United Kingdom,1,5117.03


In [13]:
# Estbalishing 99th percentile for quantity, price and revenue. Removing observations greater than or equal to 99th percentile
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM stock_code_clean
    ),
    no_cancellations AS (
        SELECT * FROM deduped
        WHERE CAST(Invoice AS VARCHAR) NOT LIKE 'C%'
    ),
    base_transactions AS (
        SELECT * 
        FROM no_cancellations
        WHERE Quantity > 0 AND Price > 0
    ),
    guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM base_transactions
    ),
    revenue_derive AS (
        SELECT 
            *,
            Quantity * Price AS Revenue
        FROM guest_tag
    ),
    no_outliers AS (
        SELECT * FROM (
            SELECT
                *,
                QUANTILE_CONT(Quantity, 0.99) OVER () AS p99_qty,
                QUANTILE_CONT(Price, 0.99) OVER () AS p99_price,
                QUANTILE_CONT(Revenue, 0.99) OVER () AS p99_revenue
            FROM
                revenue_derive
        )
        WHERE Quantity <= p99_qty
            AND Price <= p99_price
            AND Revenue <= p99_revenue
    )
    SELECT 
        p99_qty as qty_threshold,
        p99_price as price_threshold,
        p99_revenue as revenue_threshold,
        count(*) as total_rows_after_outlier_removal
    FROM no_outliers
    GROUP BY
        qty_threshold,
        price_threshold,
        revenue_threshold
    """).fetchdf()

,qty_threshold,price_threshold,revenue_threshold,total_rows_after_outlier_removal
0,108.0,18.0,184.588,984825


# 7. Date Features

In [14]:
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    no_cancellations AS (
        SELECT * FROM deduped
        WHERE CAST(Invoice AS VARCHAR) NOT LIKE 'C%'
    ),
    base_transactions AS (
        SELECT * 
        FROM no_cancellations
        WHERE Quantity > 0 AND Price > 0
    ),
    guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM base_transactions
    ),
    revenue_derive AS (
        SELECT 
            *,
            Quantity * Price AS Revenue
        FROM guest_tag
    ),
    no_outliers AS (
        SELECT * FROM (
            SELECT
                *,
                QUANTILE_CONT(Quantity, 0.99) OVER () AS p99_qty,
                QUANTILE_CONT(Price, 0.99) OVER () AS p99_price,
                QUANTILE_CONT(Revenue, 0.99) OVER () AS p99_revenue
            FROM
                revenue_derive
        )
        WHERE Quantity <= p99_qty
            AND Price <= p99_price
            AND Revenue <= p99_revenue
    ),
    cleaned AS (
        SELECT *,
            YEAR(InvoiceDate) AS Year,
            MONTH(InvoiceDate) AS Month,
            DAYOFWEEK(InvoiceDate) AS DayOfWeek,
            HOUR(InvoiceDate) AS Hour,
            CASE WHEN DAYOFWEEK(InvoiceDate) IN (0, 6) THEN 1 ELSE 0 END AS IsWeekend,
            DATE_TRUNC('week',  InvoiceDate) AS WeekStart,
            DATE_TRUNC('month', InvoiceDate) AS MonthStart
        FROM no_outliers
    )
    SELECT * FROM cleaned LIMIT 5;
    """).fetchdf()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,is_guest,Revenue,p99_qty,p99_price,p99_revenue,Year,Month,DayOfWeek,Hour,IsWeekend,WeekStart,MonthStart
0,528613,22664,TOY TIDY DOLLY GIRL DESIGN,1,2010-10-22 15:57:00,4.21,NaN,United Kingdom,1,4.21,108.0,18.0,184.588,2010,10,5,15,0,2010-10-18,2010-10-01
1,528613,22680,"FRENCH BLUE METAL DOOR SIGN, 5",1,2010-10-22 15:57:00,2.51,NaN,United Kingdom,1,2.51,108.0,18.0,184.588,2010,10,5,15,0,2010-10-18,2010-10-01
2,528613,22791,T-LIGHT GLASS FLUTED ANTIQUE,3,2010-10-22 15:57:00,2.51,NaN,United Kingdom,1,7.53,108.0,18.0,184.588,2010,10,5,15,0,2010-10-18,2010-10-01
3,528613,35004B,SET OF 3 BLACK FLYING DUCKS,2,2010-10-22 15:57:00,5.91,NaN,United Kingdom,1,11.82,108.0,18.0,184.588,2010,10,5,15,0,2010-10-18,2010-10-01
4,528613,90018C,SILVER BLACK ORBIT DROP EARRINGS,1,2010-10-22 15:57:00,4.24,NaN,United Kingdom,1,4.24,108.0,18.0,184.588,2010,10,5,15,0,2010-10-18,2010-10-01


# 8. Validate Final Dataset

In [15]:
# Counting number of observations per data cleaning step

con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    no_cancellations AS (
        SELECT * FROM deduped
        WHERE CAST(Invoice AS VARCHAR) NOT LIKE 'C%'
    ),
    base_transactions AS (
        SELECT * 
        FROM no_cancellations
        WHERE Quantity > 0 AND Price > 0
    ),
    guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM base_transactions
    ),
    revenue_derive AS (
        SELECT 
            *,
            Quantity * Price AS Revenue
        FROM guest_tag
    ),
    no_outliers AS (
        SELECT * FROM (
            SELECT
                *,
                QUANTILE_CONT(Quantity, 0.99) OVER () AS p99_qty,
                QUANTILE_CONT(Price, 0.99) OVER () AS p99_price,
                QUANTILE_CONT(Revenue, 0.99) OVER () AS p99_revenue
            FROM
                revenue_derive
        )
        WHERE Quantity <= p99_qty
            AND Price <= p99_price
            AND Revenue <= p99_revenue
    ),
    cleaned AS (
        SELECT *,
            YEAR(InvoiceDate) AS Year,
            MONTH(InvoiceDate) AS Month,
            DAYOFWEEK(InvoiceDate) AS DayOfWeek,
            HOUR(InvoiceDate) AS Hour,
            CASE WHEN DAYOFWEEK(InvoiceDate) IN (0, 6) THEN 1 ELSE 0 END AS IsWeekend,
            DATE_TRUNC('week',  InvoiceDate) AS WeekStart,
            DATE_TRUNC('month', InvoiceDate) AS MonthStart
        FROM no_outliers
    )
    SELECT 'raw' AS stage, COUNT(*) AS num_of_rows FROM raw_retail
    UNION ALL SELECT 'deduped', COUNT(*) FROM deduped
    UNION ALL SELECT 'no_cancellations', COUNT(*) FROM no_cancellations
    UNION ALL SELECT 'base_transactions', COUNT(*) FROM base_transactions
    UNION ALL SELECT 'with_guest_tag', COUNT(*) FROM guest_tag
    UNION ALL SELECT 'with_revenue', COUNT(*) FROM revenue_derive
    UNION ALL SELECT 'no_outliers', COUNT(*) FROM no_outliers
    UNION ALL SELECT 'cleaned', COUNT(*) FROM cleaned;
    """).fetchdf()

,stage,num_of_rows
0,raw,1067371
1,deduped,1033036
2,no_cancellations,1013932
3,base_transactions,1007913
4,with_guest_tag,1007913
5,with_revenue,1007913
6,no_outliers,984825
7,cleaned,984825


In [16]:
# Checking if any data quality issues still exist
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    no_cancellations AS (
        SELECT * FROM deduped
        WHERE CAST(Invoice AS VARCHAR) NOT LIKE 'C%'
    ),
    base_transactions AS (
        SELECT * 
        FROM no_cancellations
        WHERE Quantity > 0 AND Price > 0
    ),
    guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM base_transactions
    ),
    revenue_derive AS (
        SELECT 
            *,
            Quantity * Price AS Revenue
        FROM guest_tag
    ),
    no_outliers AS (
        SELECT * FROM (
            SELECT
                *,
                QUANTILE_CONT(Quantity, 0.99) OVER () AS p99_qty,
                QUANTILE_CONT(Price, 0.99) OVER () AS p99_price,
                QUANTILE_CONT(Revenue, 0.99) OVER () AS p99_revenue
            FROM
                revenue_derive
        )
        WHERE Quantity <= p99_qty
            AND Price <= p99_price
            AND Revenue <= p99_revenue
    ),
    cleaned AS (
        SELECT *,
            YEAR(InvoiceDate) AS Year,
            MONTH(InvoiceDate) AS Month,
            DAYOFWEEK(InvoiceDate) AS DayOfWeek,
            HOUR(InvoiceDate) AS Hour,
            CASE WHEN DAYOFWEEK(InvoiceDate) IN (0, 6) THEN 1 ELSE 0 END AS IsWeekend,
            DATE_TRUNC('week',  InvoiceDate) AS WeekStart,
            DATE_TRUNC('month', InvoiceDate) AS MonthStart
        FROM no_outliers
    )
    SELECT
        SUM(CASE WHEN Quantity <= 0 THEN 1 ELSE 0 END) AS bad_quantity,
        SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS bad_price,
        SUM(CASE WHEN Revenue <= 0 THEN 1 ELSE 0 END) AS bad_revenue,
        SUM(CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' THEN 1 ELSE 0 END) AS cancellations_remain,
        SUM(CASE WHEN IsWeekend NOT IN (0, 1) THEN 1 ELSE 0 END) AS bad_weekend_flag,
        SUM(CASE WHEN is_guest  NOT IN (0, 1) THEN 1 ELSE 0 END) AS bad_guest_flag
    FROM cleaned;
    """).fetchdf()

,bad_quantity,bad_price,bad_revenue,cancellations_remain,bad_weekend_flag,bad_guest_flag
0,0.0,0.0,0.0,0.0,0.0,0.0


# 9. Saving cleaned data

In [17]:
con.execute("""
    CREATE OR REPLACE VIEW cleaned AS
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    no_cancellations AS (
        SELECT * FROM deduped
        WHERE CAST(Invoice AS VARCHAR) NOT LIKE 'C%'
    ),
    base_transactions AS (
        SELECT * 
        FROM no_cancellations
        WHERE Quantity > 0 AND Price > 0
    ),
    guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM base_transactions
    ),
    revenue_derive AS (
        SELECT 
            *,
            Quantity * Price AS Revenue
        FROM guest_tag
    ),
    no_outliers AS (
        SELECT * FROM (
            SELECT
                *,
                QUANTILE_CONT(Quantity, 0.99) OVER () AS p99_qty,
                QUANTILE_CONT(Price, 0.99) OVER () AS p99_price,
                QUANTILE_CONT(Revenue, 0.99) OVER () AS p99_revenue
            FROM
                revenue_derive
        )
        WHERE Quantity <= p99_qty
            AND Price <= p99_price
            AND Revenue <= p99_revenue
    ),
    cleaned AS (
        SELECT *,
            YEAR(InvoiceDate) AS Year,
            MONTH(InvoiceDate) AS Month,
            DAYOFWEEK(InvoiceDate) AS DayOfWeek,
            HOUR(InvoiceDate) AS Hour,
            CASE WHEN DAYOFWEEK(InvoiceDate) IN (0, 6) THEN 1 ELSE 0 END AS IsWeekend,
            DATE_TRUNC('week',  InvoiceDate) AS WeekStart,
            DATE_TRUNC('month', InvoiceDate) AS MonthStart
        FROM no_outliers
    )
    SELECT *,
        YEAR(InvoiceDate) AS Year,
        MONTH(InvoiceDate) AS Month,
        DAYOFWEEK(InvoiceDate) AS DayOfWeek,
        HOUR(InvoiceDate) AS Hour,
        CASE WHEN DAYOFWEEK(InvoiceDate) IN (0, 6) THEN 1 ELSE 0 END AS IsWeekend,
        DATE_TRUNC('week',  InvoiceDate) AS WeekStart,
        DATE_TRUNC('month', InvoiceDate) AS MonthStart
    FROM no_outliers;
""")

In [18]:
# All valid sales data including guests
con.execute("""
    COPY (SELECT * FROM cleaned)
    TO '../data/processed/cleaned.csv' (HEADER, DELIMITER ',')
""")

In [19]:
# Registered customers only
con.execute("""
    COPY (SELECT * FROM cleaned WHERE is_guest = 0)
    TO '../data/processed/registered_only.csv' (HEADER, DELIMITER ',');
""")

# 10. Cleaning Summary

In [20]:
stats = con.execute("""
    SELECT
        COUNT(*) AS final_rows,
        COUNT(DISTINCT CustomerID) AS unique_customers,
        COUNT(DISTINCT StockCode) AS unique_products,
        COUNT(DISTINCT Country) AS unique_countries,
        ROUND(SUM(Revenue), 2) AS total_revenue
    FROM cleaned
""").fetchdf()

raw_rows    = len(raw)
final_rows  = stats['final_rows'][0]
removed     = raw_rows - final_rows
pct_removed = removed / raw_rows * 100
customers   = stats['unique_customers'][0]
products    = stats['unique_products'][0]
countries   = stats['unique_countries'][0]
revenue     = stats['total_revenue'][0]

print('=' * 60)
print('CLEANING SUMMARY')
print('=' * 60)
print(f"""
ROWS:
  Original:  {raw_rows:,}
  Final:     {final_rows:,}
  Removed:   {removed:,} ({pct_removed:.1f}%)

WHAT WAS REMOVED AND WHY:
  • Exact row duplicated, likely a data extraction issue as same transaction can't have happened twice or more at the same time down to the second.
  • Cancellations are they are not sales, observations with 'C' prefix in invoice were removed
  • Negative quantity removed because these were returns and reversals
  • Zero price removed as these were bad debt entries and negative price removed due to no clear meaning to them
  • Outliers (>p99) removed, probably bulk orders which are unrepresentative of typical retail customer behaviour

FINAL DATASET (cleaned.csv):
  - Rows: {final_rows:,}
  - Customers: {customers:,} (registered only, guests excluded from count)
  - Products: {products:,}
  - Countries: {countries:,}
  - Total Revenue: GBP {revenue:,.2f}

OUTPUT FILES:
  • cleaned.csv ---> all valid sales including guests
  • registered_only.csv ---> valid sales with CustomerID for cohort and RFM

Ready for analysis.ipynb
""")

CLEANING SUMMARY

ROWS:
  Original:  1,067,371
  Final:     984,825
  Removed:   82,546 (7.7%)

WHAT WAS REMOVED AND WHY:
  • Exact row duplicated, likely a data extraction issue as same transaction can't have happened twice or more at the same time down to the second.
  • Cancellations are they are not sales, observations with 'C' prefix in invoice were removed
  • Negative quantity removed because these were returns and reversals
  • Zero price removed as these were bad debt entries and negative price removed due to no clear meaning to them
  • Outliers (>p99) removed, probably bulk orders which are unrepresentative of typical retail customer behaviour

FINAL DATASET (cleaned.csv):
  - Rows: 984,825
  - Customers: 5,780 (registered only, guests excluded from count)
  - Products: 4,847
  - Countries: 43
  - Total Revenue: GBP 14,625,928.52

OUTPUT FILES:
  • cleaned.csv ---> all valid sales including guests
  • registered_only.csv ---> valid sales with CustomerID for cohort and RFM

R